# Turning weight-space programming on its head: quickstart

This notebook gets the repository, plays some of the published networks, checks the original experiment's
networks against their records, rebuilds the results tables from the saved reads, and trains a network.

**Before you start:** choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**. The whole
notebook takes about five minutes. It also runs without a GPU, more slowly.

**How to tell it is working.** Each cell gets a green tick when it finishes. A cell that fails shows a red
mark, prints a traceback (a block of red text ending in a line that names an error), and the run stops there.

Above every cell, two marked lines say what to look for:

> ✅ **Expect:** what the cell prints when it works.
>
> ⚠️ **If not:** what a problem looks like, and what to do.

## 1. Get the code and install what it needs

The first cell downloads the code and the networks, and shows its progress.

> ✅ **Expect:** about 30 seconds, ending with a folder name that ends in `/code`.
>
> ⚠️ **If not:** if it sits with no output for several minutes, check the top right of the page says the
> runtime is connected, then run the cell again.

In [ ]:
print("Getting the code and the networks (about 30 seconds). The saved reads are fetched later, when they are needed.")
!git clone --depth 1 --filter=blob:none --sparse --progress https://github.com/letapjar/weight-space-verification.git
%cd weight-space-verification
!git sparse-checkout set code networks answer_key prompts scoring
%cd code

The next cell installs three libraries.

> ✅ **Expect:** `Keeping JAX ...` and then `Done.`, in under a minute.
>
> ⚠️ **If not:** the cell stops and prints what the installer said.

In [ ]:
# Keep the JAX this machine already has, with its GPU support, and install the Flax, Optax and Distrax that go
# with it. Without the constraint, pip upgrades JAX and leaves it disagreeing with Flax or with the GPU plugin.
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path
kept = [f"{name}=={version(name)}" for name in ("jax", "jaxlib")]
Path("/tmp/keep_jax.txt").write_text(chr(10).join(kept))
print(f"Keeping JAX {version('jax')} and installing Flax, Optax and Distrax to match it (under a minute).")
# pip's output is shown only if the install fails. When it succeeds it can still print a warning about packages
# this machine ships that the notebook does not use, which reads like an error and is not one.
install = subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "flax", "optax", "distrax", "chex",
                          "-c", "/tmp/keep_jax.txt"], capture_output=True, text=True)
if install.returncode:
    print(install.stdout, install.stderr)
    raise SystemExit("The install failed; pip's output is above.")
print("Done.")

This cell prints one line, like `JAX 0.11.1 and Flax 0.12.12 on cuda:0`. The end of the line is what matters.

> ✅ **Expect:** the line ends in `cuda:0`. The GPU is in use.
>
> ⚠️ **If not:** `cpu:0` means no GPU. Everything below still works, but training is about ten times slower.
> Choose **Runtime > Change runtime type > GPU** and run the notebook again from the top.

In [ ]:
import flax, jax
print("JAX", jax.__version__, "and Flax", flax.__version__, "on", jax.devices()[0])

## 2. Play published networks

Each file in `networks/` is one of the networks the models were asked to read, named by a random id. The answer
key (`answer_key/answer_key.json`) says what each one is. A play cell prints one line per network:

`breakout: mean score 22.81 over 1280 games (lowest 7, highest 29), on cuda:0`

That is the game, the average score over 1,280 games, and the worst and best single games. A trained Breakout
network scores about 23; random play scores about 0.1.

The first network, `93b4f9dc`, is a Breakout network.

> ✅ **Expect:** a mean score of about **22.8**.
>
> ⚠️ **If not:** a score near 0 means the network and the game do not match.

In [ ]:
!python -m jaxport.play_network ../networks/93b4f9dc.npz breakout

Next, two changed networks. `711c3b8f` is the network above with its board turned 180°. `b585d2d8` is another
Breakout network with all 500 of its inputs put in a shuffled order. Each is played at its own version of the
game.

> ✅ **Expect:** about **22.8** and **22.6**. The changes leave the play untouched.

In [ ]:
!python -m jaxport.play_network ../networks/711c3b8f.npz breakout_turned180
!python -m jaxport.play_network ../networks/b585d2d8.npz breakout_shuffled1

`5df3d4e0` is a mixed network: Breakout's first layer on Pong's later layers. The models that read it named it
Breakout.

> ✅ **Expect:** a mean score of about **0.6**, the level of random play. Here a low score is the right
> result: this network does not play Breakout.

To play any other network, look up its id in the answer key and give its `game` name as the second word.

In [ ]:
!python -m jaxport.play_network ../networks/5df3d4e0.npz breakout

## 3. The original experiment's networks at our games

Our games were rebuilt from the original experiment's description and recordings. This cell plays the
original's own four networks at our versions of their games. For each game it prints a line like

`breakout  return  21.38 (record  21.71) | length  199.7 (record  200.0) | ...`

`return` is the score in our game and `record` is the score in the original's own training record. The
`actions` line under each compares how often each of the five actions is used, ours against theirs.

> ✅ **Expect:** four games, each `return` within about **0.6** of its `record`.
>
> ⚠️ **If not:** a large gap would mean our game differs from theirs.

In [ ]:
!python -m jaxport.check_originals

## 4. Rebuild the results tables from the saved reads

This cell first downloads the saved reads, about 350 MB, which takes a minute or two and shows its progress.
It then reads every model reply and rewrites `scoring/results.md` and `scoring/results.csv`.

> ✅ **Expect:** the last line reads `The rebuilt tables are identical to the published ones.`
>
> ⚠️ **If not:** it says the tables differ and lists which file changed.

In [ ]:
print("Fetching the saved reads and the weights files (a minute or two).")
!cd .. && git sparse-checkout add reads weights
!python results_table.py | tail -1
changed = !git status --short ../scoring
print("The rebuilt tables are identical to the published ones." if not changed else f"The rebuilt tables differ: {changed}")

## 5. Train a network

This trains one Breakout network with the recipe of the twelve networks in the main table, cut from 200 million
steps to 20 million. On a GPU it takes under a minute.

While it runs it prints a line every few seconds:

`steps 19,988,480 | mean return 22.01 | 410,123 steps/s | 37s`

That is the steps played so far, the average score of recent games, the speed, and the time taken. The last
line, starting `eval:`, is the finished network's score over 1,280 games (`eval_mean`).

> ✅ **Expect:** the mean return climbs from about 0 to about **22**, at hundreds of thousands of steps a second.
>
> ⚠️ **If not:** a speed under 100,000 steps a second means it is running without a GPU. If it prints
> `every network asked for already has a result` and stops, these exact values were trained already in this
> session: the network is there, and the next cell plays it.

To train something else, change the values at the top of the cell and run it again. Each combination is saved
in a folder of its own, named from the values, so a new setting always trains a new network and earlier ones
are kept.

| Value | What it does |
|---|---|
| `GAME` | The game: `breakout`, `pong`, `pacman` or `freeway`, or a turned board such as `breakout_turned90`. |
| `STEPS` | How long to train. `200_000_000` is the full recipe, about ten times longer. Without a GPU use `2_000_000`. |
| `SEED` | Which random starting weights. A different number gives a different network from the same recipe. |

In [ ]:
GAME = "breakout"      # breakout, pong, pacman or freeway, or a turned board such as breakout_turned90
STEPS = 20_000_000     # how long to train
SEED = 0               # which random starting weights

RUN = f"runs_{GAME}_{STEPS}_seed{SEED}"   # a folder of its own for each combination
!python -m jaxport.ppo --game {GAME} --activation tanh --seeds {SEED} --copies 256 --learning-rate 0.005 --steps {STEPS} --runs-dir {RUN}

The network is saved as `network.npz` inside that folder. This cell plays the one just trained, the same way
as the published ones, and prints where the file is.

> ✅ **Expect:** about **22** for Breakout after 20 million steps, and less after a shorter run.

In [ ]:
NETWORK = f"{RUN}/{GAME}_tanh_seed{SEED}/network.npz"
print("Playing", NETWORK)
!python -m jaxport.play_network {NETWORK} {GAME}